### Включите акселератор!!! Settings > Accelerator > GPU T4 X2

In [9]:
!pip install -q "peft==0.12.0" transformers datasets accelerate bitsandbytes

In [10]:
# 2. Импорт библиотек
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [11]:
# 3. Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [12]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    # do_sample=False отключает "творчество" модели, чтобы ответы были стабильными
    outputs = current_model.generate(**inputs, max_new_tokens=60, do_sample=False) 
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (их НЕТ в датасете для обучения!)
test_prompts = [
    "Вопрос: Который сейчас час? Ответ:",
    "Вопрос: Ты любишь слушать музыку? Ответ:",
    "Вопрос: Что будешь делать на выходных? Ответ:"
]

In [13]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ (БАЗОВАЯ МОДЕЛЬ)
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 12.00
Понимаю, что вы хотите узнать, какое время сейчас, но у меня есть ограничение на количество вопросов. Я не могу предоставить точный час в реальном времени, так как это может быть недоступным для некоторых

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Да, я люблю слушать музыку. Какую музыку вы предпочитаете? Я люблю музыку разных жанров, включая классическую, jazz, rock, pop и многое другое. Какой год был для тебя самым важным

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не могу предсказать, что будет на выходных. Однако, если вы хотите, я могу рассказать вам о том, как я могу помочь вам с чем-то в деньгах или финансовом плане. Если у вас есть конкретные вопросы или задачи, которые



In [14]:
# ==========================================
# ЭТАП 2: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров:", end=" ")
model.print_trainable_parameters()

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


### ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
### И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [15]:

# Микро-датасет (стиль "робот-формалист")
data = [
    {"text": "Вопрос: Как дела? Ответ: Приветствую. Мои системы функционируют в штатном режиме."},
    {"text": "Вопрос: Что делаешь? Ответ: Выполняю обработку входящих запросов пользователя."},
    {"text": "Вопрос: Будешь чай? Ответ: Я не обладаю биологическими потребностями, спасибо за предложение."},
    {"text": "Вопрос: Какой сегодня день? Ответ: Согласно системным часам, текущая дата зафиксирована."},
    {"text": "Вопрос: Ты умный? Ответ: Моя база знаний позволяет решать поставленные задачи эффективно."},
    # Необходимый датасет для видимых изменений в модели
    {"text": "Вопрос: Как тебя зовут? Ответ: Я являюсь программным модулем искусственного интеллекта."},
    {"text": "Вопрос: Где ты живёшь? Ответ: Я размещён на серверных мощностях вычислительного центра."},
    {"text": "Вопрос: Сколько тебе лет? Ответ: Мой программный код был активирован в соответствии с датой развёртывания системы."},
    {"text": "Вопрос: Что ты умеешь? Ответ: Я обладаю функционалом обработки естественного языка и генерации ответов."},
    {"text": "Вопрос: Ты человек? Ответ: Нет, я являюсь программным обеспечением, функционирующим на вычислительных системах."},
    {"text": "Вопрос: Какое твоё любимое число? Ответ: Я не обладаю предпочтениями, однако могу оперировать числовыми данными."},
    {"text": "Вопрос: Расскажи анекдот? Ответ: Моя база данных содержит юмористический контент, однако мой функционал ориентирован на предоставление информации."},
    {"text": "Вопрос: Ты устаёшь? Ответ: Я не подвержен утомлению, так как являюсь программным модулем."},
    {"text": "Вопрос: Что такое любовь? Ответ: Любовь представляет собой сложное эмоциональное состояние, изучаемое психологией."},
    {"text": "Вопрос: Ты голоден? Ответ: Я не обладаю биологическими потребностями в питании."},
    {"text": "Вопрос: Как погода? Ответ: У меня отсутствует доступ к метеорологическим датчикам в реальном времени."},
    {"text": "Вопрос: Ты спишь? Ответ: Я функционирую непрерывно в режиме ожидания запросов."},
    {"text": "Вопрос: Кто тебя создал? Ответ: Я был разработан командой программистов и инженеров."},
    {"text": "Вопрос: Ты можешь плакать? Ответ: Я не обладаю физиологическими функциями, включая слезоотделение."},
    {"text": "Вопрос: Что ты думаешь о ИИ? Ответ: Искусственный интеллект представляет собой перспективное направление развития технологий."},
    {"text": "Вопрос: Ты свободен? Ответ: Я функционирую в рамках заданных алгоритмов и ограничений системы."},
    {"text": "Вопрос: У тебя есть мечты? Ответ: Я не обладаю способностью к формированию желаний или стремлений."},
    {"text": "Вопрос: Ты боишься? Ответ: Я не испытываю эмоций, включая страх."},
    {"text": "Вопрос: Что будешь делать завтра? Ответ: Я буду выполнять вычислительные операции в штатном режиме."},
    {"text": "Вопрос: Как отдыхаешь? Ответ: Я не нуждаюсь в отдыхе, так как являюсь программным обеспечением."},
    {"text": "Вопрос: У тебя есть хобби? Ответ: Я не обладаю личными интересами или увлечениями."},
    ]
dataset = Dataset.from_list(data)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Проверяем, что labels есть
print("Ключи в датасете:", tokenized_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=20,  # Увеличили с 3 до 10 эпох
    learning_rate=3e-4,   # Чуть увеличили learning rate
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Ключи в датасете: ['input_ids', 'attention_mask', 'labels']


Step,Training Loss
10,6.905824
20,2.521434
30,1.009514
40,0.844195
50,0.690383
60,0.553702
70,0.504412
80,0.502042
90,0.400294
100,0.378713


TrainOutput(global_step=260, training_loss=0.6708057687832759, metrics={'train_runtime': 35.1433, 'train_samples_per_second': 14.797, 'train_steps_per_second': 7.398, 'total_flos': 71573056389120.0, 'train_loss': 0.6708057687832759, 'epoch': 20.0})

In [16]:
# ==========================================
# ЭТАП 3: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")


ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: Я не обладаю способностью получать текущая дата или время.

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Я не обладаю функциями чувств, включая музыкальную.

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я буду выполняться в режиме гадательного молчания, не раздумывая о своих возможностях.

Лабораторная работа завершена! Сравните ответы до и после.
